## Simulating Data
We use a Maximum-Likelihood Multivariate Auto-Regression Model to generate data that we then employ. This generation of data requires two steps:
1. Creating a ML-MAR model - usually using pre-existing data. We obtain the necessary coefficients that then define this ML-MAR model.
2. Simulating data from the ML-MAR model - we use the obtained coefficients to simulate new data that can be used in the analysis later on.

Afterwards, we need to randomly flip a couple of channels for each subject to create what we call the "ambiguous data".

#### 1. Creating ML-MAR 
This step is also skippable. A .npy file is provided with already-computed coefficients that can help simulate data further.

#### 2. Simulating Data from ML-MAR
Using the obtained coefficients and a function, we will now simulate the data.

First we load the coefficients from the matlab file.

In [4]:
import numpy as np
import pandas as pd

In [1]:
import scipy.io as sio

filename = 'mlmar_coeffs_25_ch.mat'  # specify the name of the folder and file
data = sio.loadmat(filename)

W = data['W']  # specify the data structure in which the coeffs are located
covm = data['covm'] # specify the covariance matrix structure
pred = data['pred']  # specify the predicted values

T_new = pred.shape[0] # this let's us know how many data series are present and the number of data points for each series
X_init = None

In [2]:
from simulate_from_mlmar import sim_mlmar
X = sim_mlmar(W, T_new, X_init)

However, we need to generate data that consists of multiple subjects and has the same number of channels for each subject file. Let's do that next.

Note: The data from which we generated our ML-MAR coefficients consisted of 10 , 25 or 50 channels, so that would stay consistent (on the basis of which coefficient file you choose) regardless of how many subject files you generate.

In [7]:
from scipy.io import savemat

no_patients = 10  # specify here the number of subject files you want to generate
x_ref_data = {}  # dict to save the data for all subjects here so we can use it later on
for i in range(no_patients):
    # simulate ml-mar and obtain simulated data for the current patient
    X = sim_mlmar(W, T_new, X_init)
    
    # save this simulated data in a .mat file (or a .npy file)
    dic = {"data":X, "nsamples":T_new}
    filename = "unflipped_data_subject_" + str(i+1)+ ".mat"
    savemat(filename, dic)

    # save the data in the dict
    df = pd.DataFrame(X)
    x_ref_data[i] = df

Now you would be able to see 10 new files generated in your current folder.

What we need to do next is to ambiguously flip the data in some of the channels randomly for each subject. We keep the information about which channels we flipped saved, so that later on we can evaluate the solution of our algorithm against our 'ground-truth' data.

In [12]:
prob_flip = 0.5  # specify the limit - if the probability of flipping a channel 
# is lower than this, the channel should be flipped

# randomly flipping the signs of some of the channels
no_patients = len(x_ref_data)
no_channels = x_ref_data[0].shape[1]
x_amb_data = {}

# we make an array that holds the probability of flipping each channel for each subject
probability_flip = np.random.rand(no_patients, no_channels)

# holds bool whether the channel should be flipped acc to the prob
do_flip = np.zeros([no_patients, no_channels])

for i in range(no_patients):
    X = x_ref_data[i].to_numpy()
    for j in range(no_channels):
        if probability_flip[i,j] < prob_flip:
            do_flip[i,j] = 1
            X[:, j] = -X[:, j]  # flip the channel

    df = pd.DataFrame(X) # store the flipped data for the current subject
    x_amb_data[i] = df
    
    # save the data in a .mat file
    T_new = X.shape[0]
    dic = {"data":X, "nsamples":T_new}
    filename = "ambiguous_data_subject_" + str(i+1) + ".mat"
    savemat(filename, dic)
    

In [13]:
pd.DataFrame(do_flip)  # [subjects x channels]

,0,1,2,3,4,5,6,7,8,9
0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0
2,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
3,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0
4,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
5,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0
7,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0
8,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0
9,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0
